# PyTorch

A refresher on **PyTorch** — the define-by-run tensor + autograd library that most modern deep-learning research and a large slice of production runs on.

**Domain:** AI/ML Tooling  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**PyTorch is a numerical computing library for tensors (n-dimensional arrays) with two superpowers bolted on:** GPU acceleration and **automatic differentiation**. On top of those two primitives it ships a neural-network library (`torch.nn`), optimizers (`torch.optim`), data loading (`torch.utils.data`), and a graph compiler (`torch.compile`).

**The problem it solves.** Training a neural net means computing gradients of a scalar loss with respect to millions of parameters, then nudging the parameters downhill. Doing that by hand is error-prone and doesn't scale. PyTorch records every operation you perform on tensors into a dynamic graph and, on `loss.backward()`, walks that graph in reverse applying the chain rule — so you only ever write the *forward* pass and get gradients for free.

**Why it won.** PyTorch is **define-by-run** (eager): the graph is built as your Python executes, so a model is just normal Python with normal control flow, normal debugging (`pdb`, `print`), and normal stack traces. That immediacy is why it dominates research, and why the ecosystem above it — Hugging Face Transformers, Lightning, torchvision/torchaudio, diffusers — is PyTorch-first.

**Reach for it when** you're building or fine-tuning neural networks, need autograd over custom math, want GPU/accelerator compute with a NumPy-like API, or you're consuming the PyTorch model ecosystem (almost every open-weights LLM ships as PyTorch).

**Don't reach for it when** classical ML (trees, linear models, SVMs) fits — use [scikit-learn](scikit-learn.ipynb). For pure array math with no learning, NumPy is lighter. For heavily TPU-centric or functional/JIT-first work, [JAX](jax-flax.ipynb) is a strong alternative.

## 2. Mental Model

**A `Tensor` is a NumPy array that remembers where it came from.**

Every tensor created with `requires_grad=True` (and every tensor derived from one) carries a hidden pointer — `.grad_fn` — back to the operation that produced it. Chaining operations therefore builds a **computation graph** behind the scenes, leaf-to-root, as your forward pass runs:

```
x ──┐
     (matmul) ── h ── (relu) ── a ── (mse) ── loss
W ──┘
```

When you call `loss.backward()`, autograd traverses that graph **in reverse**, multiplying local derivatives (the chain rule) and accumulating the result into each leaf tensor's `.grad`. The optimizer then reads those `.grad` values and updates the parameters. Three moves, every training step:

1. **Forward** — run data through the model → `loss` (graph is recorded).
2. **Backward** — `loss.backward()` fills every parameter's `.grad`.
3. **Step** — `optimizer.step()` applies the update; `optimizer.zero_grad()` clears grads for next time.

The graph is **rebuilt from scratch on every forward pass** (that's "dynamic" / define-by-run), which is exactly why `if`/`for`/recursion in your model "just work" — there's no static graph to pre-compile.

## 3. Key Concepts

- **Tensor** — the core data type: an n-d array on a `device` (`cpu`, `cuda`, `mps`) with a `dtype` (`float32`, `bfloat16`, …). NumPy-like API (`reshape`, broadcasting, indexing) plus `.to(device)`.
- **Autograd** — the reverse-mode automatic differentiation engine. `requires_grad=True` opts a tensor in; `.grad_fn` records the op; `.backward()` computes gradients; `.grad` stores them.
- **`nn.Module`** — base class for models/layers. Holds **parameters** (learnable tensors registered automatically) and a `forward()` method. Compose modules to build networks; `model.parameters()` yields everything to optimize.
- **Optimizer (`torch.optim`)** — owns the update rule (SGD, Adam, AdamW). You hand it `model.parameters()`; it reads `.grad` and mutates the params on `.step()`.
- **Loss function** — a scalar-valued module/function (`nn.MSELoss`, `nn.CrossEntropyLoss`) whose `.backward()` seeds the gradient flow.
- **`Dataset` / `DataLoader`** — abstractions for batching, shuffling, and parallel data loading.
- **`zero_grad()`** — gradients **accumulate** by default (they add into `.grad`), so you must clear them each step or batches will contaminate each other.
- **`torch.no_grad()` / `eval()`** — context that disables graph recording (for inference/validation, saves memory) vs `model.eval()`, which switches layers like dropout/batchnorm into eval behavior. Different things — you usually want both at inference.
- **Device & dtype discipline** — tensors in one operation must share a device; mismatches are the #1 runtime error.
- **`state_dict`** — a plain dict of a model's tensors; the canonical thing you save/load for checkpoints.
- **`torch.compile`** — (2.x) a JIT that traces and fuses your model into optimized kernels for a speedup, while you still write eager code.

## 4. Setup

PyTorch is `pip`-installable, but the **right wheel depends on your hardware**. The CPU-only build is small and works everywhere; for an NVIDIA GPU you pick a CUDA-matched wheel from the official selector.

```bash
# CPU-only (what this notebook uses — small, portable):
pip install torch --index-url https://download.pytorch.org/whl/cpu

# NVIDIA GPU (example: CUDA 12.4) — see https://pytorch.org/get-started/locally/
pip install torch --index-url https://download.pytorch.org/whl/cu124
```

In a notebook you'd typically run `%pip install torch --index-url https://download.pytorch.org/whl/cpu`. The cell below just verifies the install and reports the available device.

In [1]:
import torch

# Reproducibility: seed the global RNG so this notebook's numbers are stable.
torch.manual_seed(0)

# Pick the best available device without assuming a GPU exists.
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"PyTorch version : {torch.__version__}")
print(f"Using device    : {device}")

# A tensor is a NumPy-like array that lives on a device and knows its dtype.
x = torch.arange(6, dtype=torch.float32).reshape(2, 3)
print("\nTensor x:")
print(x)
print(f"shape={tuple(x.shape)}  dtype={x.dtype}  device={x.device}")

PyTorch version : 2.12.1
Using device    : mps

Tensor x:
tensor([[0., 1., 2.],
        [3., 4., 5.]])
shape=(2, 3)  dtype=torch.float32  device=cpu


## 5. Worked Examples

Two self-contained, CPU-friendly examples:

1. **Autograd from first principles** — differentiate a scalar function by hand-checking PyTorch's gradient.
2. **Train a tiny model** — fit a linear regression with `nn.Module` + an optimizer, the canonical forward/backward/step loop.

### Example 1 — Autograd: gradients for free

Take `y = x²·sin(x)` at `x = 2.0`. By calculus the derivative is `dy/dx = 2x·sin(x) + x²·cos(x)`. PyTorch should match it without us ever writing the derivative.

In [2]:
import math

x = torch.tensor(2.0, requires_grad=True)   # opt this leaf into autograd
y = x**2 * torch.sin(x)                      # graph is recorded as we go

y.backward()                                 # walk the graph in reverse → fills x.grad

# Hand-computed derivative for comparison.
manual = 2 * 2.0 * math.sin(2.0) + 2.0**2 * math.cos(2.0)

print(f"y               = {y.item():.6f}")
print(f"x.grad (autograd) = {x.grad.item():.6f}")
print(f"manual derivative = {manual:.6f}")
print(f"match: {math.isclose(x.grad.item(), manual, rel_tol=1e-5)}")

y               = 3.637190
x.grad (autograd) = 1.972602
manual derivative = 1.972602
match: True


### Example 2 — Train a tiny linear model

Generate data from a known line `y = 3x + 2` (plus noise), then let PyTorch *recover* the slope and intercept via gradient descent. This is the full training loop in miniature: **forward → loss → `zero_grad` → `backward` → `step`**.

In [3]:
import torch.nn as nn

torch.manual_seed(0)

# Synthetic data from a known relationship: y = 3x + 2 + noise.
X = torch.linspace(-1, 1, 100).unsqueeze(1)        # shape (100, 1)
true_w, true_b = 3.0, 2.0
y = true_w * X + true_b + 0.1 * torch.randn_like(X)

# Model: a single linear layer (1 input -> 1 output) has exactly a weight and bias.
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(200):
    pred = model(X)              # forward pass (records graph)
    loss = loss_fn(pred, y)      # scalar loss

    optimizer.zero_grad()        # clear grads — they accumulate otherwise!
    loss.backward()              # backward pass — fills every param's .grad
    optimizer.step()             # apply the update

    if epoch % 50 == 0 or epoch == 199:
        print(f"epoch {epoch:3d}  loss={loss.item():.4f}")

w = model.weight.item()
b = model.bias.item()
print(f"\nlearned: y = {w:.3f}x + {b:.3f}   (target: y = {true_w}x + {true_b})")

epoch   0  loss=7.4358
epoch  50  loss=0.0136
epoch 100  loss=0.0105
epoch 150  loss=0.0105
epoch 199  loss=0.0105

learned: y = 3.000x + 2.004   (target: y = 3.0x + 2.0)


### Inference and saving — the patterns you'll repeat

At inference, wrap the forward pass in `torch.no_grad()` (skip graph building → less memory, faster) and call `model.eval()` (switches dropout/batchnorm to eval mode). Checkpointing saves the `state_dict`, not the Python object.

In [4]:
# Inference: no_grad disables autograd bookkeeping; eval() sets layer modes.
model.eval()
with torch.no_grad():
    test_x = torch.tensor([[0.5]])
    print(f"prediction at x=0.5: {model(test_x).item():.3f}  (expected ~{true_w*0.5 + true_b:.3f})")

# Checkpointing: save/load the state_dict (a plain dict of tensors), not the object.
state = model.state_dict()
print("\nstate_dict keys:", list(state.keys()))

reloaded = nn.Linear(1, 1)
reloaded.load_state_dict(state)            # restore weights into a fresh model
print("reloaded weight matches:", torch.equal(reloaded.weight, model.weight))

prediction at x=0.5: 3.504  (expected ~3.500)

state_dict keys: ['weight', 'bias']
reloaded weight matches: True


## 6. Gotchas & Pitfalls

- **Forgetting `optimizer.zero_grad()`.** Gradients **accumulate** into `.grad` by default. Skip the clear and each step trains on the sum of all previous batches' gradients — loss diverges or behaves bizarrely. (The accumulation is *intentional* — it's how you simulate large batches across micro-steps — but you must opt in, not out.)
- **Device mismatch.** `tensor on cpu` + `tensor on cuda` raises `RuntimeError: Expected all tensors to be on the same device`. Move data *and* model with `.to(device)`. Remember `.to()` on a tensor returns a **new** tensor (it's not in-place), whereas on a `Module` it mutates in place.
- **Calling `.item()` / `.numpy()` inside the training loop** forces a CPU sync and can silently wreck GPU throughput. Keep things as tensors; only pull scalars out for logging.
- **`model.eval()` vs `torch.no_grad()` are different.** `eval()` changes *layer behavior* (dropout off, batchnorm uses running stats); `no_grad()` changes *autograd* (no graph, less memory). At inference you almost always want **both**. Forgetting `eval()` is a classic "my val accuracy is noisy/wrong" bug.
- **Accumulating loss as a tensor** (`total += loss` instead of `total += loss.item()`) keeps the whole graph alive across iterations → memory leak / OOM.
- **In-place ops on tensors that need grad** (e.g. `x += 1` on a graph leaf) can corrupt the backward pass — autograd will raise about a modified-in-place variable. Prefer out-of-place when in doubt.
- **`requires_grad` vs `nn.Parameter`.** Tensors assigned as plain attributes on a module are *not* tracked as parameters; wrap learnable tensors in `nn.Parameter` (or use `nn.` layers) so `model.parameters()` finds them.
- **Saving the whole model object** via `torch.save(model, ...)` pickles your class and breaks on refactor. Save the `state_dict` instead.
- **Non-determinism.** Seeding `torch.manual_seed` isn't always enough on GPU; some cuDNN kernels are nondeterministic unless you set `torch.use_deterministic_algorithms(True)`.

## 7. When to Use vs Alternatives

| Tool | Use it when | Trade-off vs PyTorch |
|------|-------------|----------------------|
| **PyTorch** | Deep learning research & most production; custom autograd; consuming the open-weights/HF ecosystem | The baseline. Eager-first, biggest ecosystem, you write the loop yourself |
| **[PyTorch Lightning](pytorch-lightning.ipynb)** | You want PyTorch but tired of writing the boilerplate training loop, multi-GPU, checkpointing | A framework *on top of* PyTorch — less control, more structure |
| **[Keras](keras.ipynb) / [TensorFlow](tensorflow.ipynb)** | High-level `model.fit()` API; TF Serving / TFLite / mobile deployment story | Smaller research mindshare today; Keras 3 now runs on multiple backends incl. PyTorch |
| **[JAX/Flax](jax-flax.ipynb)** | Functional purity, `jit`/`vmap`/`grad` composability, TPU-first, large-scale research | Steeper learning curve, smaller ecosystem, functional style is unfamiliar |
| **[scikit-learn](scikit-learn.ipynb)** | Classical ML — trees, linear/logistic, SVM, clustering on tabular data | Not deep learning; no autograd/GPU training of nets |
| **NumPy** | Pure array math, no learning/gradients needed | No autograd, no GPU, no nn layers |

**Rule of thumb:** neural network involved → PyTorch (reach for Lightning once the boilerplate annoys you). Tabular/classical → scikit-learn. TPU-scale or functional-style research → JAX.

## 8. Resources

- **Official docs** — <https://pytorch.org/docs/stable/index.html> (the API reference; autograd, nn, optim).
- **Get-started / install selector** — <https://pytorch.org/get-started/locally/> (pick the right CUDA/CPU wheel).
- **"Deep Learning with PyTorch: A 60 Minute Blitz"** — <https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html> (the canonical fast on-ramp).
- **Autograd mechanics** — <https://pytorch.org/docs/stable/notes/autograd.html> (how the graph and `.backward()` actually work).
- **`torch.compile` tutorial** — <https://pytorch.org/tutorials/intermediate/torch_compile_tutorial.html> (the 2.x speedup path).
- **Related notebooks in this library:** [PyTorch Lightning](pytorch-lightning.ipynb), [Hugging Face](huggingface.ipynb), [JAX/Flax](jax-flax.ipynb), [scikit-learn](scikit-learn.ipynb).